# Semana 5: Ejercicios - Facts, Dimensions, SCDs y Construccion del Modelo

**Bootcamp:** Fundamentos de Ingenieria de Datos - Databricks SQL  
**Bloque tematico:** MODELAR | Practica complementaria  
**Instructor:** Luciano Argolo | www.LucianoArgolo.com


## Prerequisito

Tener `bootcamp.silver.propiedades` poblada (resultado de Semana 3-4).

| Concepto | Que es | Ejemplo |
|----------|--------|---------|
| **Surrogate Key (SK)** | Clave artificial generada por el DW (IDENTITY). Inmutable, para JOINs | `zona_id = 42` en dim_zona |
| **Business Key (BK)** | Clave natural del negocio. Para matching en MERGE e identificar la entidad | `partido + region` en dim_zona |

En SCD2: mismo BK puede tener multiples SKs (una por version). El SK identifica la version, el BK identifica la entidad.

### Verificacion inicial

In [0]:
%sql
SELECT COUNT(*) AS total_registros FROM bootcamp.silver.propiedades;

In [0]:
%sql
DESCRIBE TABLE bootcamp.silver.propiedades;

---
# MODULO 1: Facts vs Dims - Clasificacion, Conceptos y SCDs

---

## Ejercicio 1.1: Clasificar campos como Fact o Dimension

**Objetivo:** Aplicar el criterio de cardinalidad y tipo de dato para separar metricas de atributos descriptivos.

Ejecuta una query que calcule `COUNT(DISTINCT campo)` para los campos clave. Despues, clasifica cada uno.

In [0]:
%sql
SELECT
  COUNT(DISTINCT partido) AS dist_partido,
  COUNT(DISTINCT tipo_operacion) AS dist_tipo_op,
  COUNT(DISTINCT estado) AS dist_estado,
  COUNT(DISTINCT moneda) AS dist_moneda,
  COUNT(DISTINCT precio) AS dist_precio,
  COUNT(DISTINCT metros_cuadrados_totales) AS dist_m2,
  COUNT(DISTINCT url) AS dist_url,
  COUNT(*) AS total_filas
FROM bootcamp.silver.propiedades;

### Respuesta 1.1: Clasificacion

| Campo | Cardinalidad | Clasificacion | Justificacion |
|-------|-------------|---------------|---------------|
| `partido` | Baja (~65) | **Dimension** | Categorico, baja cardinalidad, agrupa registros |
| `tipo_operacion` | Muy baja (~9) | **Dimension** | Categorico, pocas categorias |
| `estado` | Baja (~20) | **Dimension** | Categorico, describe condicion del inmueble |
| `moneda` | Muy baja (~3-4) | **Degenerate dim / Dimension** | Tan pocos valores que puede ir en la fact o en una dim minima |
| `precio` | Alta | **Metrica (Fact)** | Numerico, se suma/promedia |
| `metros_cuadrados_totales` | Alta | **Metrica (Fact)** | Numerico, se suma/promedia |
| `url` | Casi unica por fila | **Degenerate Dimension** | Cardinalidad ~ total de filas, no justifica tabla propia, pero sirve como trazabilidad |

`url` es una **degenerate dimension**: vive en la fact table porque tiene cardinalidad altisima (casi 1:1 con las filas) y no tiene atributos descriptivos propios. Sirve para trazar cada registro a la publicacion original.

---
## Ejercicio 1.2: Degenerate Dimension - \u00bfcuando SI y cuando NO?

**Objetivo:** Entender los limites del concepto y aplicar criterio propio.

**Pregunta:** Tu equipo discute si `moneda` deberia ser una degenerate dimension (ir directo en la fact) o tener su propia tabla `dim_moneda`. Solo hay 3-4 valores (ARS, USD, MXN, GUARANIES), pero podrias querer agregar atributos como `nombre_completo`, `simbolo`, `pais_origen`.

### Respuesta 1.2

**Con 3-4 valores sin atributos extras:** Degenerate dimension. No tiene sentido crear una tabla, un schema, un JOIN para 3 valores que son solo codigos. Va directo en la fact como `moneda STRING`.

**Si en el futuro se agregan 20 monedas con metadatos** (nombre completo, simbolo, pais de origen, tipo de cambio base): conviene crear `dim_moneda` porque ahora la moneda tiene **atributos descriptivos propios** que enriquecen el analisis.

**Criterio:** No es solo cardinalidad. La pregunta clave es: \u00bfnecesito atributos descriptivos extras sobre esta entidad? Si la respuesta es si -> dimension. Si es solo un codigo plano que no cambia -> degenerate dimension.

---
## Ejercicio 1.3: SK vs BK - \u00bfPor que necesitamos ambas?

**Objetivo:** Razonar sobre la necesidad de dos tipos de claves en un modelo dimensional.

**Pregunta:** Imagina que `dim_zona` solo tiene `partido` y `region` como PK compuesta (Business Key pura, sin Surrogate Key). \u00bfQue problemas surgen cuando: (a) un partido cambia de nombre, (b) necesitas hacer SCD2, (c) tu fact_propiedades tiene que joinear por 2 campos en vez de 1?

### Respuesta 1.3

Si `dim_zona` usa solo `partido + region` como PK compuesta (sin surrogate key):

**(a) Cambio de nombre:** Si "capital-fedral" se corrige a "capital-federal", la BK cambia. Todas las filas en `fact_propiedades` que referenciaban "capital-fedral" quedan huerfanas porque la FK ya no matchea con la nueva BK. Con una SK (`zona_id = 42`), el ID nunca cambia — se actualiza el atributo `partido` en la dim y las facts siguen apuntando al mismo `zona_id`.

**(b) SCD2:** En SCD2 necesitas multiples versiones del mismo registro. Si la PK es `partido + region`, no podes tener dos filas con la misma combinacion. Con `zona_id` IDENTITY, cada version tiene su propio SK unico (ej: zona_id=42 version vieja, zona_id=107 version nueva), ambas con `partido='capital-federal'`.

**(c) JOINs compuestos:** Joinear por 2 campos (`ON sp.partido = dz.partido AND sp.region = dz.region`) es mas lento y propenso a errores que joinear por 1 campo numerico (`ON fp.zona_id = dz.zona_id`). Con SK, el JOIN es siempre 1 campo, mas rapido y mas simple.

---
## Ejercicio 1.4: Arbol de decision SCD - Elegir el tipo correcto

**Objetivo:** Aplicar el arbol de decision SCD a escenarios reales.

Para cada caso, indica SCD Type 1, 2 o 3 y justifica.

### Respuesta 1.4

| Caso | SCD Type | Justificacion |
|------|----------|---------------|
| (a) Corregir typo "capital-fedral" -> "capital-federal" | **SCD Type 1** | Es una correccion de error, no un cambio de negocio. No necesitamos saber que alguna vez estuvo mal escrito. Sobrescribimos. |
| (b) Partido cambia de region (GBA Norte -> GBA Oeste), necesitas historico | **SCD Type 2** | Necesitas saber a que region pertenecia en cada momento para reportes historicos. Se crea nueva version con `valid_from`/`valid_to`. |
| (c) Barrio cambia de nombre, solo necesitas el anterior | **SCD Type 3** | Solo necesitas "antes y despues", no todo el historial. Se agrega columna `nombre_anterior` y se actualiza en la misma fila. |
| (d) Reclasificar tipo de operacion, reportes viejos deben reflejar el nuevo nombre | **SCD Type 1** | Si los reportes viejos deben mostrar el nombre nuevo, entonces el pasado se "reescribe". No me importa preservar el valor anterior. |

**Regla rapida:** SCD1 = no me importa el pasado. SCD2 = necesito todas las versiones. SCD3 = solo necesito "antes y despues".

---
## Ejercicio 1.5: SCD Type 1 - Overwrite con MERGE

**Objetivo:** Implementar una correccion que sobreescriba el valor anterior sin guardar historico.

Sobre `bootcamp.gold.dim_zona`, crear una vista staging con datos corregidos y aplicar MERGE.

In [0]:
%sql
-- Paso 1: Vista staging con correcciones
CREATE OR REPLACE TEMP VIEW staging_correcciones AS
SELECT partido, region, 'Gran Buenos Aires' AS ciudad
FROM bootcamp.gold.dim_zona
WHERE region = 'gba norte' AND ciudad = 'GBA';

select * from staging_correcciones

In [0]:
%sql
-- Verificar COUNT antes del MERGE
SELECT COUNT(*) AS count_antes FROM bootcamp.gold.dim_zona;

In [0]:
%sql
-- Paso 2: MERGE SCD1 (solo UPDATE, nunca INSERT de segunda version)
MERGE INTO bootcamp.gold.dim_zona AS target
USING staging_correcciones AS source
ON target.partido = source.partido AND target.region = source.region
WHEN MATCHED THEN
  UPDATE SET ciudad = source.ciudad;

In [0]:
%sql
-- Paso 3: Verificar que no se crearon filas nuevas
SELECT COUNT(*) AS count_despues FROM bootcamp.gold.dim_zona;

---
## Ejercicio 1.6: SCD Type 2 - Crear tabla con columnas de historizacion

**Objetivo:** Disenar y crear una dimension preparada para SCD Type 2.

In [0]:
%sql
-- Paso 1: DDL con columnas de control SCD2
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd2 (
  zona_id       BIGINT GENERATED ALWAYS AS IDENTITY,
  partido       STRING NOT NULL,
  region        STRING NOT NULL,
  ciudad        STRING,
  provincia     STRING DEFAULT 'Buenos Aires',
  pais          STRING DEFAULT 'Argentina',
  valid_from    TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  valid_to      TIMESTAMP DEFAULT '9999-12-31',
  is_current    BOOLEAN DEFAULT TRUE
) USING DELTA;

In [0]:
%sql
-- Paso 2: Carga inicial desde Silver
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad)
SELECT DISTINCT partido, region,
  CASE WHEN region = 'capital federal' THEN 'CABA' ELSE 'GBA' END
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL;

In [0]:
%sql
-- Paso 3: Verificar que todos son is_current = TRUE
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN is_current = TRUE THEN 1 ELSE 0 END) AS actuales,
  SUM(CASE WHEN is_current = FALSE THEN 1 ELSE 0 END) AS historicos
FROM bootcamp.gold.dim_zona_scd2;

---
## Ejercicio 1.7: SCD Type 2 - Historizar un cambio (manual)

**Objetivo:** Ejecutar el proceso manual de cerrar version anterior + insertar version nueva.

**Regla:** Siempre cerrar primero, insertar despues. Si insertas primero, el UPDATE cierra la fila recien creada.

In [0]:
%sql
-- Paso 1: CERRAR la version actual de 'capital federal'
-- IMPORTANTE: el BK en Silver es 'capital federal' (con espacio), no 'capital-federal' (guion)
UPDATE bootcamp.gold.dim_zona_scd2
SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHERE partido = 'capital federal' AND region = 'capital federal' AND is_current = TRUE;

In [0]:
%sql
-- Paso 2: INSERTAR nueva version (mismo BK, distinto valor de ciudad)
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad, valid_from, valid_to, is_current)
VALUES ('capital federal', 'capital federal', 'Buenos Aires', CURRENT_TIMESTAMP(), '9999-12-31', TRUE);

In [0]:
%sql
-- Paso 3: Verificar - debe mostrar 2 filas (version cerrada + version nueva)
SELECT zona_id, partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido = 'capital federal' AND region = 'capital federal'
ORDER BY valid_from;

---
## Ejercicio 1.8: SCD Type 2 - Historizar con MERGE (masivo)

**Objetivo:** Automatizar SCD2 con MERGE para actualizar multiples registros a la vez.

El MERGE solo puede hacer UPDATE o INSERT en una pasada. Para SCD2 completo (cerrar + insertar nueva version), necesitas 2 operaciones.

In [0]:
%sql
-- Paso 1: Vista staging con cambios simulados
CREATE OR REPLACE TEMP VIEW staging_cambios_scd2 AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA Sur'),
  ('barrio-nuevo', 'capital federal', 'CABA')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql
-- Paso 2: MERGE para CERRAR registros que cambiaron
MERGE INTO bootcamp.gold.dim_zona_scd2 AS target
USING staging_cambios_scd2 AS source
ON target.partido = source.partido
   AND target.region = source.region
   AND target.is_current = TRUE
WHEN MATCHED AND target.ciudad != source.ciudad_nueva THEN
  UPDATE SET valid_to = CURRENT_TIMESTAMP(), is_current = FALSE
WHEN NOT MATCHED THEN
  INSERT (partido, region, ciudad, valid_from, valid_to, is_current)
  VALUES (source.partido, source.region, source.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE);

In [0]:
%sql
-- Paso 3: INSERT nuevas versiones para los que se cerraron
INSERT INTO bootcamp.gold.dim_zona_scd2 (partido, region, ciudad, valid_from, valid_to, is_current)
SELECT s.partido, s.region, s.ciudad_nueva, CURRENT_TIMESTAMP(), '9999-12-31', TRUE
FROM staging_cambios_scd2 s
INNER JOIN bootcamp.gold.dim_zona_scd2 t
  ON s.partido = t.partido AND s.region = t.region
  AND t.is_current = FALSE AND t.valid_to >= DATE_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 MINUTE)
WHERE NOT EXISTS (
  SELECT 1 FROM bootcamp.gold.dim_zona_scd2 t2
  WHERE t2.partido = s.partido AND t2.region = s.region AND t2.is_current = TRUE
);

In [0]:
%sql
-- Paso 4: Verificar - palermo debe tener 2 versiones, barrio-nuevo 1
SELECT partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido IN ('palermo', 'barrio-nuevo')
ORDER BY partido, valid_from;

---
## Ejercicio 1.9: SCD Type 3 - Guardar valor anterior con MERGE

**Objetivo:** Implementar SCD3 donde se guarda el valor previo de un atributo sin crear filas nuevas.

`ciudad_anterior = target.ciudad` guarda el valor actual *antes* de sobreescribirlo. Solo se conserva el ultimo cambio.

In [0]:
%sql
-- Paso 1: DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd3 (
  zona_id          BIGINT GENERATED ALWAYS AS IDENTITY,
  partido          STRING NOT NULL,
  region           STRING NOT NULL,
  ciudad           STRING,
  ciudad_anterior  STRING,
  fecha_ultima_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  provincia        STRING DEFAULT 'Buenos Aires',
  pais             STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (zona_id)
)
USING DELTA
COMMENT 'Dimension de zonas -- SCD Type 3 (valor actual + anterior)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- Paso 2: Carga inicial (sin valor anterior)
INSERT INTO bootcamp.gold.dim_zona_scd3 (partido, region, ciudad)
SELECT DISTINCT partido, region,
  CASE WHEN region = 'capital federal' THEN 'CABA' ELSE 'GBA' END
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL;

In [0]:
%sql
-- Paso 3: Simular cambios
CREATE OR REPLACE TEMP VIEW source_cambios AS
SELECT * FROM VALUES
  ('palermo', 'capital federal', 'CABA Sur'),
  ('belgrano', 'capital federal', 'CABA Norte')
AS t(partido, region, ciudad_nueva);

In [0]:
%sql
-- Paso 3b: MERGE - detectar cambio y mover valor actual a anterior
MERGE INTO bootcamp.gold.dim_zona_scd3 AS target
USING source_cambios AS source
ON target.partido = source.partido AND target.region = source.region
WHEN MATCHED AND target.ciudad != source.ciudad_nueva THEN
  UPDATE SET
    ciudad_anterior = target.ciudad,
    ciudad = source.ciudad_nueva,
    fecha_ultima_actualizacion = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
  INSERT (partido, region, ciudad)
  VALUES (source.partido, source.region, source.ciudad_nueva);

In [0]:
%sql
-- Paso 4: Verificar - solo los que cambiaron tienen ciudad_anterior
SELECT partido, region, ciudad, ciudad_anterior, fecha_ultima_actualizacion
FROM bootcamp.gold.dim_zona_scd3
WHERE ciudad_anterior IS NOT NULL
ORDER BY partido;

---
## Ejercicio 1.10: Trade-off - \u00bfSCD2 para todo?

**Objetivo:** Pensar criticamente sobre cuando SCD2 es overkill.

Tu empresa tiene 5 dimensiones: `dim_zona` (65 valores), `dim_tipo_operacion` (9 valores), `dim_caracteristicas` (~40 valores), `dim_orientacion` (~15 valores), `dim_tiempo` (fechas). \u00bfA cuales les aplicarias SCD2 y a cuales SCD1?

### Respuesta 1.10

| Dimension | Valores | \u00bfCambia? | \u00bfNecesitan historico? | Decision | Justificacion |
|-----------|---------|----------|--------------------------|----------|---------------|
| `dim_zona` | ~65 | Si, partidos pueden cambiar de region | Si, reportes historicos | **SCD2** | Cambio real de negocio con impacto en reportes |
| `dim_tipo_operacion` | ~9 | Casi nunca | No | **SCD1** | Si cambia, es una correccion de nombre. 9 valores que no cambian no justifican el overhead de SCD2 |
| `dim_caracteristicas` | ~40 | Raramente | No | **SCD1** | Combinaciones estado+cochera estables. Cambios serian correcciones |
| `dim_orientacion` | ~15 | Casi nunca | No | **SCD1** | Valores estables de orientacion. No cambian |
| `dim_tiempo` | N fechas | **Nunca** | N/A | **Sin SCD (estatica)** | 2024-01-15 siempre es lunes. Las fechas no cambian |

**El 80% de las dims en la industria son SCD1.** SCD2 solo se justifica cuando: (a) el atributo realmente cambia, (b) alguien necesita el historico, y (c) el costo de mantenerlo vale la pena.

---
# MODULO 2: Crear Dimensiones desde Silver + Fact Table

---

## Ejercicio 2.1: dim_zona - DDL + ETL

**Objetivo:** Crear la dimension geografica con SK + BK y poblarla desde Silver.

In [0]:
%sql
-- DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
  zona_id    BIGINT GENERATED ALWAYS AS IDENTITY,
  partido    STRING NOT NULL,
  region     STRING NOT NULL,
  ciudad     STRING,
  provincia  STRING DEFAULT 'Buenos Aires',
  pais       STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (zona_id)
)
USING DELTA
COMMENT 'Dimension de zonas geograficas -- BK: partido + region'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- ETL
INSERT INTO bootcamp.gold.dim_zona (partido, region, ciudad)
SELECT DISTINCT partido, region,
  CASE WHEN region = 'capital federal' THEN 'CABA' ELSE 'GBA' END
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL;

In [0]:
%sql
-- Verificar (~65 zonas esperadas)
SELECT COUNT(*) AS total_zonas FROM bootcamp.gold.dim_zona;

---
## Ejercicio 2.2: dim_tipo_operacion + dim_caracteristicas - DDL + ETL

**Objetivo:** Crear dos dimensiones mas siguiendo el mismo patron DDL + ETL.

In [0]:
%sql
-- dim_tipo_operacion DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion (
  tipo_operacion_id  BIGINT GENERATED ALWAYS AS IDENTITY,
  tipo_operacion     STRING NOT NULL,
  categoria          STRING,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (tipo_operacion_id)
)
USING DELTA
COMMENT 'Dimension de tipos de operacion'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- dim_tipo_operacion ETL
INSERT INTO bootcamp.gold.dim_tipo_operacion (tipo_operacion, categoria)
SELECT DISTINCT
  tipo_operacion,
  CASE
    WHEN tipo_operacion LIKE '%temporal%' THEN 'temporal'
    WHEN tipo_operacion IN ('alquiler', 'venta') THEN 'residencial'
    ELSE 'otro'
  END AS categoria
FROM bootcamp.silver.propiedades
WHERE tipo_operacion IS NOT NULL;

In [0]:
%sql
-- dim_caracteristicas DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas (
  caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  estado             STRING NOT NULL,
  cochera            BOOLEAN,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Dimension de caracteristicas de la propiedad (estado + cochera) - Star Schema'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- dim_caracteristicas ETL
INSERT INTO bootcamp.gold.dim_caracteristicas (estado, cochera)
SELECT DISTINCT
  COALESCE(estado, 'sin especificar') AS estado,
  COALESCE(cochera, false) AS cochera
FROM bootcamp.silver.propiedades;

In [0]:
%sql
-- Verificar conteos
SELECT 'dim_tipo_operacion' AS dim, COUNT(*) AS total FROM bootcamp.gold.dim_tipo_operacion
UNION ALL
SELECT 'dim_caracteristicas', COUNT(*) FROM bootcamp.gold.dim_caracteristicas;

---
## Ejercicio 2.3: dim_tiempo - DDL + ETL

**Objetivo:** Crear una dimension de tiempo a partir de fechas de procesamiento.

In [0]:
%sql
-- DDL
CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tiempo (
  fecha_id          BIGINT NOT NULL,
  fecha             DATE,
  anio              INT,
  mes               INT,
  trimestre         INT,
  dia_semana        STRING,
  es_fin_de_semana  BOOLEAN,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
  PRIMARY KEY (fecha_id)
)
USING DELTA
COMMENT 'Dimension de tiempo -- fecha_id formato YYYYMMDD'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
-- ETL desde fechas de _processing_timestamp
INSERT INTO bootcamp.gold.dim_tiempo (fecha_id, fecha, anio, mes, trimestre, dia_semana, es_fin_de_semana)
SELECT DISTINCT
  CAST(DATE_FORMAT(CAST(_processing_timestamp AS DATE), 'yyyyMMdd') AS BIGINT) AS fecha_id,
  CAST(_processing_timestamp AS DATE) AS fecha,
  YEAR(_processing_timestamp) AS anio,
  MONTH(_processing_timestamp) AS mes,
  QUARTER(_processing_timestamp) AS trimestre,
  DATE_FORMAT(_processing_timestamp, 'EEEE') AS dia_semana,
  CASE WHEN DAYOFWEEK(_processing_timestamp) IN (1, 7) THEN TRUE ELSE FALSE END AS es_fin_de_semana
FROM bootcamp.silver.propiedades
WHERE _processing_timestamp IS NOT NULL;

In [0]:
%sql
-- Verificar
SELECT COUNT(*) AS total_fechas, MIN(fecha) AS fecha_min, MAX(fecha) AS fecha_max
FROM bootcamp.gold.dim_tiempo;

---
## Ejercicio 2.4: Verificar todas las dimensiones

**Objetivo:** Confirmar que las 5 dimensiones se crearon correctamente.

In [0]:
%sql
SELECT 'dim_zona' AS dim_nombre, COUNT(*) AS total_registros, COUNT(*) - COUNT(partido) AS nulls_bk
FROM bootcamp.gold.dim_zona
UNION ALL
SELECT 'dim_tipo_operacion', COUNT(*), COUNT(*) - COUNT(tipo_operacion)
FROM bootcamp.gold.dim_tipo_operacion
UNION ALL
SELECT 'dim_caracteristicas', COUNT(*), COUNT(*) - COUNT(estado)
FROM bootcamp.gold.dim_caracteristicas
UNION ALL
SELECT 'dim_orientacion', COUNT(*), COUNT(*) - COUNT(orientacion)
FROM bootcamp.gold.dim_orientacion
UNION ALL
SELECT 'dim_tiempo', COUNT(*), COUNT(*) - COUNT(fecha)
FROM bootcamp.gold.dim_tiempo;

---
## Ejercicio 2.5: fact_propiedades - DDL con row_hash

**Objetivo:** Crear la fact table central con PK deterministica y FKs a todas las dims.

`url` en la fact es una **degenerate dimension**: no justifica su propia tabla porque tiene cardinalidad altisima, pero sirve para trazar cada fila a la publicacion original.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bootcamp.gold.fact_propiedades (
  row_hash                    STRING NOT NULL,
  zona_id                     BIGINT,
  tipo_operacion_id           BIGINT,
  fecha_id                    BIGINT,
  caracteristicas_id          BIGINT,
  orientacion_id              BIGINT,
  precio                      DECIMAL(15,2),
  expensas                    DECIMAL(15,2),
  precio_por_m2               DECIMAL(15,2),
  metros_cuadrados_totales    DECIMAL(15,2),
  metros_cuadrados_cubiertos  DECIMAL(15,2),
  ambientes                   INT,
  url                         STRING,
  _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING delta
COMMENT 'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)'
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

---
## Ejercicio 2.6: fact_propiedades - ETL con LEFT JOINs

**Objetivo:** Poblar la fact joineando Silver con cada dimension para obtener las SKs.

LEFT JOIN para no perder registros si alguna dim no tiene match.

In [0]:
%sql
INSERT INTO bootcamp.gold.fact_propiedades (
  row_hash, zona_id, tipo_operacion_id, fecha_id,
  caracteristicas_id, orientacion_id,
  precio, expensas, precio_por_m2,
  metros_cuadrados_totales, metros_cuadrados_cubiertos,
  ambientes, url
)
SELECT
  MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
  dz.zona_id,
  dt.tipo_operacion_id,
  CAST(DATE_FORMAT(sp.fecha_publicacion, 'yyyyMMdd') AS BIGINT) AS fecha_id,
  dc.caracteristicas_id,
  do.orientacion_id,
  sp.precio,
  sp.expensas,
  sp.precio_por_m2,
  sp.metros_cuadrados_totales,
  sp.metros_cuadrados_cubiertos,
  sp.ambientes,
  sp.url
FROM bootcamp.silver.propiedades sp
LEFT JOIN bootcamp.gold.dim_zona dz
  ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt
  ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
LEFT JOIN bootcamp.gold.dim_caracteristicas dc
  ON COALESCE(sp.estado, 'sin especificar') = dc.estado
  AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN bootcamp.gold.dim_orientacion do
  ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LEFT JOIN bootcamp.gold.dim_tiempo dtm
  ON sp.fecha_publicacion = dtm.fecha;

In [0]:
%sql
-- Verificar: total debe coincidir con Silver
SELECT
  (SELECT COUNT(*) FROM bootcamp.silver.propiedades) AS silver_count,
  (SELECT COUNT(*) FROM bootcamp.gold.fact_propiedades) AS fact_count;

---
## Ejercicio 2.7: Verificar integridad referencial

**Objetivo:** Detectar FKs nulas y entender por que existen.

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(zona_id) AS con_zona_id,
  COUNT(*) - COUNT(zona_id) AS sin_zona_id,
  COUNT(tipo_operacion_id) AS con_tipo_op_id,
  COUNT(*) - COUNT(tipo_operacion_id) AS sin_tipo_op_id,
  COUNT(caracteristicas_id) AS con_caract_id,
  COUNT(*) - COUNT(caracteristicas_id) AS sin_caract_id,
  COUNT(orientacion_id) AS con_orient_id,
  COUNT(*) - COUNT(orientacion_id) AS sin_orient_id,
  COUNT(fecha_id) AS con_fecha_id,
  COUNT(*) - COUNT(fecha_id) AS sin_fecha_id
FROM bootcamp.gold.fact_propiedades;

### Interpretacion 2.7

Registros sin FK pueden deberse a:
- **(a)** Silver tiene `partido = NULL` -> no matchea con dim_zona
- **(b)** La dim no se genero para esa combinacion (ej: una combinacion partido+region que no se incluyo en el DISTINCT)
- **(c)** El JOIN condition no matchea (ej: espacios extra, case sensitivity)

Las FKs nulas **no son necesariamente un error** — son registros de Silver que no tienen match en alguna dimension. Se documentan y monitorean como metricas de calidad de datos.

---
## Ejercicio 2.8: Trade-off - \u00bfLEFT JOIN o INNER JOIN en la fact?

**Objetivo:** Razonar sobre la decision de conservar o descartar registros sin match.

In [0]:
%sql
-- Comparar LEFT JOIN vs INNER JOIN: cuantos registros se pierden?
SELECT
  (SELECT COUNT(*) FROM bootcamp.gold.fact_propiedades) AS con_left_join,
  (SELECT COUNT(*)
   FROM bootcamp.silver.propiedades sp
   INNER JOIN bootcamp.gold.dim_zona dz ON sp.partido = dz.partido AND sp.region = dz.region
   INNER JOIN bootcamp.gold.dim_tipo_operacion dt ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
   INNER JOIN bootcamp.gold.dim_caracteristicas dc ON COALESCE(sp.estado, 'sin especificar') = dc.estado AND COALESCE(sp.cochera, false) = dc.cochera
  ) AS con_inner_join;

### Respuesta 2.8

**LEFT JOIN (conservar con FK NULL):**
- No se pierde ningun registro de Silver
- Las FKs nulas se documentan como metricas de calidad
- Los dashboards pueden filtrar "sin zona asignada"
- **Preferido en Data Warehousing** porque el negocio puede necesitar que TODAS las propiedades esten en la fact

**INNER JOIN (descartar sin match):**
- Se pierden registros que no tienen dimension
- Los datos quedan mas "limpios" pero se pierde informacion
- Util cuando la calidad de datos es alta y las dims cubren el 100%

**Regla general:** En Data Warehousing se prefiere LEFT JOIN para preservar todos los hechos. Las FKs nulas se monitorean como metricas de calidad.

---
# MODULO 3: Queries Analiticas sobre el Star Schema + Cierre Medallion

---

## Ejercicio 3.1: Top 5 partidos por precio promedio/m2

**Objetivo:** Query analitica que cruce fact + dim_zona + dim_tipo_operacion.

In [0]:
%sql
SELECT
  dz.partido,
  dz.region,
  ROUND(AVG(fp.precio_por_m2), 2) AS avg_precio_m2,
  COUNT(*) AS total_propiedades
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
JOIN bootcamp.gold.dim_tipo_operacion dt ON fp.tipo_operacion_id = dt.tipo_operacion_id
WHERE dt.tipo_operacion = 'alquiler'
GROUP BY dz.partido, dz.region
ORDER BY avg_precio_m2 DESC
LIMIT 5;

---
## Ejercicio 3.2: Distribucion por tipo de operacion y categoria

In [0]:
%sql
SELECT
  dt.tipo_operacion,
  dt.categoria,
  COUNT(*) AS total,
  ROUND(AVG(fp.precio), 2) AS precio_promedio
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_tipo_operacion dt ON fp.tipo_operacion_id = dt.tipo_operacion_id
GROUP BY dt.tipo_operacion, dt.categoria
ORDER BY total DESC;

---
## Ejercicio 3.3: Evolucion mensual del precio promedio

**Objetivo:** Usar dim_tiempo para analisis temporal.

Si solo hay 1 fecha, la "evolucion" sera un solo punto -- eso es esperado con este dataset.

In [0]:
%sql
SELECT
  dtm.anio,
  dtm.mes,
  COUNT(*) AS publicaciones,
  ROUND(AVG(fp.precio), 2) AS precio_promedio
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_tiempo dtm ON fp.fecha_id = dtm.fecha_id
GROUP BY dtm.anio, dtm.mes
ORDER BY dtm.anio, dtm.mes;

---
## Ejercicio 3.4: Cruce de 3 dimensiones - partido x tipo x estado

**Objetivo:** Query que cruce multiples dimensiones simultaneamente.

Para propiedades en `region = 'capital federal'`, top 15 combinaciones.

In [0]:
%sql
SELECT
  dz.partido,
  dt.tipo_operacion,
  dc.estado,
  dc.cochera,
  COUNT(*) AS total,
  ROUND(AVG(fp.precio_por_m2), 2) AS avg_precio_m2
FROM bootcamp.gold.fact_propiedades fp
JOIN bootcamp.gold.dim_zona dz ON fp.zona_id = dz.zona_id
JOIN bootcamp.gold.dim_tipo_operacion dt ON fp.tipo_operacion_id = dt.tipo_operacion_id
JOIN bootcamp.gold.dim_caracteristicas dc ON fp.caracteristicas_id = dc.caracteristicas_id
WHERE dz.region = 'capital federal'
GROUP BY dz.partido, dt.tipo_operacion, dc.estado, dc.cochera
ORDER BY total DESC
LIMIT 15;

---
## Ejercicio 3.5: Query sobre SCD2 - version actual vs historico

**Objetivo:** Consultar una dimension SCD2 filtrando por version vigente y por punto en el tiempo.

In [0]:
%sql
-- Query 1: Version actual (is_current = TRUE)
SELECT partido, region, ciudad, valid_from, valid_to
FROM bootcamp.gold.dim_zona_scd2
WHERE is_current = TRUE
ORDER BY partido
LIMIT 10;

In [0]:
%sql
-- Query 2: Todas las versiones de 'capital federal'
SELECT zona_id, partido, region, ciudad, valid_from, valid_to, is_current
FROM bootcamp.gold.dim_zona_scd2
WHERE partido = 'capital federal' AND region = 'capital federal'
ORDER BY valid_from;

In [0]:
%sql
-- Query 3: Snapshot en un punto del tiempo (patron: valid_from <= fecha AND valid_to > fecha)
SELECT partido, region, ciudad
FROM bootcamp.gold.dim_zona_scd2
WHERE valid_from <= CURRENT_TIMESTAMP()
  AND valid_to > CURRENT_TIMESTAMP()
ORDER BY partido
LIMIT 10;

### Respuesta 3.5: 3 patrones de query SCD2

| Patron | Query | Uso |
|--------|-------|-----|
| **Version actual** | `WHERE is_current = TRUE` | Dashboards, reportes diarios (90% de las queries) |
| **Todas las versiones** | Sin filtro, `ORDER BY valid_from` | Auditoria, compliance, analisis de cambios |
| **Snapshot en fecha** | `WHERE valid_from <= fecha AND valid_to > fecha` | "Como era este dato en un momento especifico" |

**\u00bfPor que el 90% usa `is_current = TRUE`?** Porque la mayoria de los consumidores de datos quieren "la foto actual", no la pelicula completa. El historico se reserva para auditoria, regulatorio, y analisis de cambios ("\u00bfa que region pertenecia esta zona cuando se publico esta propiedad hace 6 meses?").

---
## Ejercicio 3.6: Checklist Medallion - \u00bfQue tenemos en cada capa?

**Objetivo:** Hacer un cierre consciente de la arquitectura construida en S2-S5.

### Respuesta 3.6

| Capa | Tablas que construimos | Garantias que da esta capa | Construida en |
|------|------------------------|---------------------------|---------------|
| **Bronze** | `bootcamp.bronze.properties_raw` (JSON ingestado tal cual) | Inmutable, append-only, fuente de verdad cruda. Si algo falla, reprocesamos desde aca | S2-S3 |
| **Silver** | `bootcamp.silver.propiedades` (limpia, tipada, deduplicada) | Tipos correctos, nulls manejados, campos derivados (`partido`, `region`, `precio_por_m2`), schema definido | S3-S4 |
| **Gold** | `dim_zona`, `dim_tipo_operacion`, `dim_caracteristicas`, `dim_orientacion`, `dim_tiempo`, `fact_propiedades`, `dim_zona_scd2`, `dim_zona_scd3` | Integridad referencial, metricas calculadas, modelo dimensional, SCDs, listo para BI | S4-S5 |

**\u00bfQue pasa si borras cada capa?**
- **Bronze:** NO se puede recrear sin re-ingestar del source original. Es la unica capa irrecuperable.
- **Silver:** Se puede reconstruir desde Bronze (si los ETLs son idempotentes).
- **Gold:** Se puede reconstruir desde Silver (dims con DISTINCT, fact con JOINs).

---
## Ejercicio 3.7: Pensamiento critico - \u00bfQue falta en nuestro pipeline?

**Objetivo:** Identificar gaps en la arquitectura actual antes de avanzar a S6.

### Respuesta 3.7

| Gap | Problema | Solucion (semana donde se cubre) |
|-----|----------|----------------------------------|
| **(a) Deteccion de cambios** | No sabemos si Silver cambio y hay que re-procesar Gold. Actualmente corremos todo manual. | Workflows con triggers o schedules (S8) |
| **(b) Valores nuevos en dims** | Si aparece un `tipo_operacion` nuevo en Silver, la dim no lo tiene y el LEFT JOIN deja FK = NULL. | ETL incremental con MERGE que inserte nuevos valores automaticamente |
| **(c) Metricas de calidad** | No tenemos checks automaticos: \u00bfcuantos nulls? \u00bfcuantos registros se perdieron? \u00bfrow counts ok? | Data quality framework (S7) |
| **(d) Orquestacion** | Todo se ejecuta manualmente desde un notebook. No hay scheduling, no hay alertas si falla. | Databricks Workflows / Airflow (S8) |
| **(e) Capa semantica** | Gold tiene tablas crudas. Los analistas necesitan views pre-armadas con JOINs resueltos. | Views + capa semantica (S7) |

**Proximas semanas:** S6 = PySpark (procesamiento distribuido), S7 = optimizacion Delta Lake + capa semantica + data quality, S8 = workflows + dashboards.

---
## Ejercicio 3.8: Pregunta de entrevista - Describi tu pipeline

**Objetivo:** Practicar la articulacion tecnica para entrevistas.

*"Describi un pipeline de datos que hayas construido, desde la ingesta hasta el consumo."*

### Respuesta 3.8

Construi un pipeline de datos end-to-end en Databricks usando la arquitectura Medallion (Bronze -> Silver -> Gold) sobre Delta Lake.

**Ingesta (Bronze):** Los datos vienen de un scraping de propiedades inmobiliarias en formato JSON. Se cargan en una tabla Bronze como STRING preservando el formato original, con metadata de ingestion (`_source_table`, `_ingestion_timestamp`). Bronze es inmutable y append-only.

**Transformacion (Silver):** En Silver aplico limpieza completa: casteo de tipos (STRING -> DECIMAL, INT, BOOLEAN), manejo de nulos (centinela 999 para antiguedad, COALESCE para estados), deduplicacion con ROW_NUMBER() OVER (PARTITION BY url, precio), y enriquecimiento con campos derivados como `partido` y `region` (mapeados con CASE WHEN desde valores crudos).

**Modelado (Gold):** En Gold implemento un Star Schema con 5 dimensiones (`dim_zona`, `dim_tipo_operacion`, `dim_caracteristicas`, `dim_orientacion`, `dim_tiempo`) y una fact table (`fact_propiedades`). Cada dimension usa IDENTITY como surrogate key. La fact tiene `row_hash` (MD5) como PK deterministica y LEFT JOINs a cada dim. Implemente SCD Type 1 para correcciones, SCD Type 2 para historizar cambios en `dim_zona`, y SCD Type 3 para tracking de valor anterior.

**Idempotencia:** Los ETLs de Silver usan INSERT OVERWRITE. Los de Gold usan CREATE TABLE IF NOT EXISTS + INSERT con deduplicacion. Los SCDs usan MERGE. Ejecutar N veces da el mismo resultado.

**Si tuviera que escalar:** Migraria los ETLs de SQL puro a PySpark para manejar volumenes mayores, agregaria Workflows para orquestacion automatica, y un framework de data quality para monitorear metricas en cada capa.

---
## Ejercicio Bonus: Relaciones N:N - \u00bfComo las modelarias?

**Objetivo:** Pensar criticamente sobre relaciones muchos-a-muchos en un modelo dimensional.

**Escenario:** Imagina que en el futuro una propiedad puede tener **multiples tipos de operacion simultaneos** (ej: "venta + alquiler" para el mismo listing). Actualmente `fact_propiedades` tiene una sola FK `tipo_operacion_id`, lo que asume relacion 1:N.

**Preguntas:**
1. \u00bfPor que una sola FK ya no alcanza para modelar esta relacion?
2. \u00bfQue es una **bridge table** (tabla puente) y como la usarias para resolver esto?
3. \u00bfQue impacto tiene en las queries analiticas? (pista: cuidado con contar doble)

### Respuesta Bonus

**(1)** Con una sola FK, cada fila de la fact solo puede referenciar UN tipo de operacion. Si una propiedad es "venta + alquiler", tendrias que duplicar la fila en la fact (una con FK a venta, otra con FK a alquiler), lo que duplica las metricas y corrompe los agregados.

**(2)** Una **bridge table** resuelve la relacion N:N con una tabla intermedia:

```
bridge_propiedad_tipo_operacion
- row_hash (FK -> fact_propiedades)
- tipo_operacion_id (FK -> dim_tipo_operacion)
- peso DECIMAL (ej: 0.5 si es 50% venta, 50% alquiler)
```

Esto convierte la relacion N:N en dos relaciones 1:N: fact -> bridge -> dim.

**(3)** Sin el campo `peso`, las queries que sumen precio contarian doble la propiedad. El peso permite ponderar: `SUM(fp.precio * b.peso)` en vez de `SUM(fp.precio)`. Es un patron avanzado pero frecuente en la industria.

---
# Checklist de Finalizacion

| | Competencia |
|---|---|
| \u2610 | Se clasificar campos como fact, dimension o degenerate dim usando cardinalidad |
| \u2610 | Entiendo SK vs BK y por que SCD2 necesita ambas |
| \u2610 | Puedo elegir SCD Type 1, 2 o 3 para un escenario dado y justificar |
| \u2610 | Implemente SCD1, SCD2 y SCD3 con MERGE |
| \u2610 | Cree 4 dimensiones con DDL + ETL desde Silver |
| \u2610 | Cree fact_propiedades con row_hash como PK y LEFT JOINs a cada dim |
| \u2610 | Verifique integridad referencial y entiendo por que hay FKs nulas |
| \u2610 | Escribi queries analiticas que cruzan fact con multiples dims |
| \u2610 | Puedo describir la arquitectura Medallion completa en una entrevista |
| \u2610 | Identifique que falta para que el pipeline sea production-ready |

---
*Instructor: Luciano Argolo | www.LucianoArgolo.com*